# RAG Basics with Mistral AI

![](../../images/rag.png)

Retrieval Augmented Generation (RAG) is an AI framework that synergizes the capabilities of LLMs and information retrieval systems. It's useful to answer questions or generate content leveraging external knowledge. There are two main steps in standard RAG:
1) **Retrieval**: retrieve relevant information from a knowledge base, often with the use of text embeddings stored in a vector store;
2) **Generation**: insert the relevant information to the prompt for the LLM to generate information. In this guide, we will walk through a very basic example of RAG using our models.

## RAG from Scratch

This section aims to guide you through the process of building a basic RAG from scratch. We have two goals: firstly, to offer users a comprehensive understanding of the internal workings of RAG and demystify the underlying mechanisms; secondly, to empower you with the essential foundations needed to build an RAG using the minimum required dependencies.


### Import needed packages
The first step is to install the needed packages `mistralai` and `faiss-cpu` and import the needed packages:



In [1]:
! pip install faiss-cpu mistralai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.9/78.9 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 66.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 48.7 MB/s eta 0:00:00


In [2]:
from mistralai.client import Mistral
import requests
import numpy as np
import faiss
import os

api_key= "API_KEY"
client = Mistral(api_key=api_key)

### Get data

In this very simple example, we are getting data from an essay written by Paul Graham:

In [3]:
response = requests.get('https://raw.githubusercontent.com/run-llama/llama_index/main/data/paul_graham_essay.txt')
text = response.text

Optionally, we can also save the essay to a local file:

In [4]:
with open('essay.txt', 'w') as f:
    f.write(text)

## Split document into chunks

In a RAG system, it is crucial to split the document into smaller chunks so that it's more effective to identify and retrieve the most relevant information in the retrieval process later.

In this example, we simply split our text by character, combine 2048 characters into each chunk, and we get 37 chunks.

In [5]:
chunk_size = 2048
chunks = [text[i:i + chunk_size] for i in range(0, len(text), chunk_size)]

In [6]:
len(chunks)

37

#### Considerations:
- **Chunk size**: Depending on your specific use case, it may be necessary to customize or experiment with different chunk sizes and chunk overlap to achieve optimal performance in RAG. For example, smaller chunks can be more beneficial in retrieval processes, as larger text chunks often contain filler text that can obscure the semantic representation. As such, using smaller text chunks in the retrieval process can enable the RAG system to identify and extract relevant information more effectively and accurately.  However, it's worth considering the trade-offs that come with using smaller chunks, such as increasing processing time and computational resources.
- **How to split**: While the simplest method is to split the text by character, there are other options depending on the use case and document structure. For example, to avoid exceeding token limits in API calls, it may be necessary to split the text by tokens. To maintain the cohesiveness of the chunks, it can be useful to split the text by sentences, paragraphs, or HTML headers. If working with code, it's often recommended to split by meaningful code chunks for example using an Abstract Syntax Tree (AST) parser.


### Create embeddings for each text chunk
For each text chunk, we then need to create text embeddings, which are numeric representations of the text in the vector space. **Words with similar meanings are expected to be in closer proximity or have a shorter distance in the vector space.**

To create an embedding, use Mistral's embeddings API endpoint and the embedding model `mistral-embed`. We create a `get_text_embedding` to get the embedding from a single text chunk and then we use list comprehension to get text embeddings for all text chunks.


In [7]:
def get_text_embedding(input):
    embeddings_batch_response = client.embeddings.create(
          model="mistral-embed",
          inputs=input
      )
    return embeddings_batch_response.data[0].embedding

In [8]:
text_embeddings = np.array([get_text_embedding(chunk) for chunk in chunks])

In [9]:
text_embeddings.shape

(37, 1024)

In [10]:
text_embeddings

array([[-0.03979492,  0.07733154,  0.00013709, ..., -0.01274109,
        -0.02101135, -0.00264168],
       [-0.03152466,  0.07226562,  0.02961731, ..., -0.01079559,
        -0.01189423, -0.00821686],
       [-0.05905151,  0.06112671,  0.01206207, ..., -0.0226593 ,
         0.00488663, -0.00665283],
       ...,
       [-0.05477905,  0.06890869,  0.02703857, ..., -0.02456665,
        -0.02526855, -0.02687073],
       [-0.03884888,  0.05587769,  0.04718018, ..., -0.01812744,
         0.00926208, -0.00866699],
       [-0.03048706,  0.05831909,  0.01704407, ..., -0.01620483,
        -0.01800537, -0.04415894]])

*If you want to learn more about Embeddings, we recommend taking a look at some of our cookbooks digging into the subject.*

### Load into a vector database
Once we get the text embeddings, **a common practice is to store them in a vector database for efficient processing and retrieval.** There are several vector database to choose from. In our simple example, we are using an open-source vector database Faiss, which allows for efficient similarity search.  

With Faiss, we instantiate an instance of the Index class, which defines the indexing structure of the vector database. We then add the text embeddings to this indexing structure.


In [11]:
d = text_embeddings.shape[1]
index = faiss.IndexFlatL2(d)
index.add(text_embeddings)

#### Considerations:
- **Vector database**: When selecting a vector database, there are several factors to consider including speed, scalability, cloud management, advanced filtering, and open-source vs. closed-source.

### Create embeddings for a question
Whenever users ask a question, we also need to create embeddings for this question using the same embedding models as before.


In [12]:
question = "What were the two main things the author worked on before college?"
question_embeddings = np.array([get_text_embedding(question)])
question_embeddings.shape

(1, 1024)

In [13]:
question_embeddings

array([[-0.05447388,  0.03497314,  0.0375061 , ..., -0.02786255,
        -0.00342941,  0.00316811]])

#### Considerations:
- Hypothetical Document Embeddings (HyDE): In some cases, the user’s question might not be the most relevant query to use for identifying the relevant context. Instead, it maybe more effective to generate a hypothetical answer or a hypothetical document based on the user’s query and use the embeddings of the generated text to retrieve similar text chunks.

### Retrieve similar chunks from the vector database
We can perform a search on the vector database with `index.search`, which takes two arguments: the first is the vector of the question embeddings, and the second is the number of similar vectors to retrieve. This function returns the distances and the indices of the most similar vectors to the question vector in the vector database. Then based on the returned indices, we can retrieve the actual relevant text chunks that correspond to those indices.


In [14]:
D, I = index.search(question_embeddings, k=2) # Returns the first 2 ids of the chunks with the highest similarity scores
print(I)

[[0 3]]


In [15]:
retrieved_chunk = [chunks[i] for i in I.tolist()[0]]
print(retrieved_chunk)

['\n\nWhat I Worked On\n\nFebruary 2021\n\nBefore college the two main things I worked on, outside of school, were writing and programming. I didn\'t write essays. I wrote what beginning writers were supposed to write then, and probably still are: short stories. My stories were awful. They had hardly any plot, just characters with strong feelings, which I imagined made them deep.\n\nThe first programs I tried writing were on the IBM 1401 that our school district used for what was then called "data processing." This was in 9th grade, so I was 13 or 14. The school district\'s 1401 happened to be in the basement of our junior high school, and my friend Rich Draves and I got permission to use it. It was like a mini Bond villain\'s lair down there, with all these alien-looking machines — CPU, disk drives, printer, card reader — sitting up on a raised floor under bright fluorescent lights.\n\nThe language we used was an early version of Fortran. You had to type programs on punch cards, then 

#### Considerations:
- **Retrieval methods**: There are a lot different retrieval strategies. In our example, we are showing a simple similarity search with embeddings. Sometimes when there is metadata available for the data, it’s better to filter the data based on the metadata first before performing similarity search. There are also other statistical retrieval methods like TF-IDF and BM25 that use frequency and distribution of terms in the document to identify relevant text chunks.
- **Retrieved document**: Do we always retrieve individual text chunk as it is? Not always.
    - Sometimes, we would like to include more context around the actual retrieved text chunk. We call the actual retrieve text chunk “child chunk” and our goal is to retrieve a larger “parent chunk” that the “child chunk” belongs to.
    - On occasion, we might also want to provide weights to our retrieve documents. For example, a time-weighted approach would help us retrieve the most recent document.
    - One common issue in the retrieval process is the “lost in the middle” problem where the information in the middle of a long context gets lost. Our models have tried to mitigate this issue. For example, in the passkey task, our models have demonstrated the ability to find a "needle in a haystack" by retrieving a randomly inserted passkey within a long prompt, up to 32k context length. However, it is worth considering experimenting with reordering the document to determine if placing the most relevant chunks at the beginning and end leads to improved results.
  
### Combine context and question in a prompt and generate response

Finally, we can offer the retrieved text chunks as the context information within the prompt. Here is a prompt template where we can include both the retrieved text and user question in the prompt.



In [16]:
prompt = f"""
Context information is below.
---------------------
{retrieved_chunk}
---------------------
Given the context information, answer the provided user query.
User Query: {question}
Answer:
"""

In [17]:
def run_mistral(user_message, model = "mistral-medium-latest"):
    messages = [
        {
            "role": "user", "content": user_message
        }
    ]
    chat_response = client.chat.complete(
        model=model,
        messages=messages
    )
    return chat_response.choices[0].message.content

In [18]:
run_mistral(prompt)

'The two main things the author worked on before college were **writing** and **programming**.'

#### Considerations:
- Prompting techniques: Most of the prompting techniques can be used in developing a RAG system as well. For example, we can use few-shot learning to guide the model’s answers by providing a few examples. Additionally, we can explicitly instruct the model to format answers in a certain way.


In the next sections, we are going to show you how to do a similar basic RAG with some of the popular RAG frameworks. We will start with LlamaIndex and add other frameworks in the future.


## Search Kit
For building more in-depth, production-ready RAG systems, **we recommend our Search and Agentic Search Kits**. With multiple built-in tools, they're perfect for efficiently building performant, enterprise-level solutions.

*Learn more about [Search](https://docs.mistral.ai/studio/search/search-toolkit) and [Agentic Search](https://docs.mistral.ai/studio/search/agentic-search).*